## Planner Agent

#### Importing libs

In [1]:
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import display, Markdown, HTML

import os
from pathlib import Path
import sys
sys.path.append(str(Path().resolve().parent))

from agents.research_agent import research_agent
from utils.load_prompt import load_prompt

load_dotenv()
CLIENT = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
MODEL = "gemini-2.0-flash"

#### Defining auxiliary functions

In [2]:
def pricing_agent(neighborhood: str) -> str:
    """
    
    """
    return f"""
        ## Estimativa de preço (mock)

        Ainda estamos calculando estimativas reais para **{neighborhood}**.

        Em breve será possível consultar preços médios de imóveis nessa região.
    """

#### Prompt Engineering

In [3]:
tools = [
    types.Tool(function_declarations=[research_agent]),
    types.Tool(function_declarations=[pricing_agent])
]

ValidationError: 1 validation error for Tool
function_declarations.0
  Input should be a valid dictionary or object to extract fields from [type=model_attributes_type, input_value=<function research_agent at 0x1146cd120>, input_type=function]
    For further information visit https://errors.pydantic.dev/2.12/v/model_attributes_type

In [4]:
def planner_agent(user_prompt: str) -> None:
    """
    Planner Agent: decide se o usuário quer uma análise qualitativa do bairro
    (Research Agent) ou consulta de preço (Pricing Agent) e conduz a conversa.
    """
    system_prompt = load_prompt("planner_agent_v1")

    response = CLIENT.models.generate_content(
        model = MODEL,
        contents = user_prompt,
        config = types.GenerateContentConfig(
            system_instruction = system_prompt,
            temperature=0.7
        )
    )

    decision_text = response.text.lower()

    if "research" in decision_text:
        output = research_agent(user_prompt)
        display(Markdown(output))

    elif "pricing" in decision_text:
        display(Markdown(f"💰 Chamando Pricing Agent para: {user_prompt}"))
        
    else:
        display(Markdown("Não entendi se você quer análise de bairro ou preço do imóvel."))

#### Testing the prompt

In [6]:
planner_agent("Queria entender se vale investir na Vila Mariana")

Não entendi se você quer análise de bairro ou preço do imóvel.